# **Qué es el análisis de sentimiento**

Es una tarea de clasificación de texto. Dado un texto, el modelo decide si la intención utilizada es positiva, negativa o neutral.

No es "**entender**" el texto como lo haría una persona. Es detectar patrones de palabras y contexto que, estadísticamente, se asocian a cada categoría.

Para esta práctica en español, el modelo más adecuado es **RoBERTuito**. Es un modelo entrenado específicamente con **500 millones de tweets en español**, lo que le permite entender jerga, emojis, abreviaturas y texto informal.

Está diseñado para redes sociales, que es donde el análisis de sentimiento tiene más aplicación real. Y funciona bien en tareas de sentimiento, odio e ironía en español.

RoBERTuito no se entrenó desde cero. Es un modelo preentrenado: alguien invirtió muchísimo tiempo y recursos en entrenarlo con textos masivos para que "entienda" el español.

Luego ese modelo base se ajustó (mediante fine-tuning) con un conjunto más pequeño de tweets etiquetados como positivos, negativos o neutros. El resultado es un modelo especializado en análisis de sentimiento en español, listo para usar.

Esto es importante porque explica por qué tú no entrenas nada. Usas un modelo que ya existe, ya está entrenado, y solo lo llamas desde Colab. Es exactamente lo que haría un desarrollador profesional: **no reinventar la rueda, usar lo que ya funciona**.

# **Hugging Face**

Hugging Face es la plataforma donde viven estos modelos. Su función **pipeline** es una abstracción: tú le dices "quiero realizar un análisis de sentimiento" y "usa este modelo", y ella se encarga de todo lo demás (preprocesar el texto, pasarlo por el modelo, devolver el resultado).

Solo se necesita entender que pipeline es el "botón" que conecta el texto con un modelo ya entrenado.

El modelo devuelve dos cosas:

    Una etiqueta: POS, NEG o NEU

    Una confianza (score), un número entre 0 y 1

La confianza no es lo mismo que **"acierto"**. Es la seguridad del modelo en su propia predicción. Un score de 0.99 no garantiza que tenga razón; un score de 0.51 en una frase ambigua es normal y no es un fallo.

El modelo no lee intenciones, lee patrones. El guiño o el signo de interrogación alteran el contexto que el modelo aprendió durante el preentrenamiento, y eso mueve la predicción.

In [1]:
from transformers import pipeline

clasificador = pipeline("sentiment-analysis", model="pysentimiento/robertuito-sentiment-analysis")
# carga un modelo de inteligencia artificial
# entrenado para analizar el sentimiento
# de textos en español, con especial enfoque
# en lenguaje informal y redes sociales

# === este es el texto  que se clasifica ===
texto = "Me encanta este módulo de IA"
# =====================================

resultado = clasificador(texto)
print(resultado)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[{'label': 'POS', 'score': 0.9514580368995667}]


In [2]:
resultado = clasificador("Esto es horrible.")
print(resultado)

[{'label': 'NEG', 'score': 0.9667719602584839}]


In [3]:
# clasifica un texto que sea negativo

texto = "Este curso es un desastre, no he aprendido nada"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEG', 'score': 0.9850068688392639}]


In [4]:
# clasifica un texto neutro
texto = "La clase empieza a las nueve en el aula 3"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.8928263783454895}]


In [5]:
# clasifica un texto irónico
texto = "Qué bien, otra vez sin internet el día de la entrega"
resultado = clasificador(texto)
print(resultado)

[{'label': 'NEU', 'score': 0.5852790474891663}]


In [6]:
# clasifica un texto usando emojis
texto = "Hoy me ha salido todo genial 😍🎉"
resultado = clasificador(texto)
print(resultado)

[{'label': 'POS', 'score': 0.9735262393951416}]


**¿Cómo funciona el análisis de sentimientos? ¿Qué límites tiene?**

Le das una frase al modelo y te devuelve una etiqueta (POS, NEG o NEU) y un score entre 0 y 1. No entiende el texto como una persona, solo detecta palabras y contexto que aprendió al entrenarse con los tweets. Con frases claras funciona bien: la negativa dio NEG (0.98), la neutra NEU (0.89) y la de emojis POS (0.97).

Sus límites se ven con la frase irónica, que el modelo marcó como NEU cuando en realidad es negativa. Tiene palabras positivas “qué bien”, pero la intención es quejarse y eso no lo capta.El score no es lo mismo que acertar, solo indica lo seguro que está el modelo y puede fallar con textos formales porque se entrenó con tweets.

# **Práctica**

Implementa en python un analizador de comentarios. Los comentarios positivos y neutros serán aceptados, salvo que contengan alguna palabra de la lista negra de palabras. Los comentarios negativos que contengan alguna palabra de la lista negra o que tengan un score de más de 0.5 serán rechazadas.

4.1 Crea la lista negra y una función que mire si el comentario contiene alguna palabra:

In [7]:
lista_negra = ["idiota", "estafa", "basura"]

def tiene_palabra_mala(comentario):
    comentario = comentario.lower()
    for palabra in lista_negra:
        if palabra in comentario:
            return True
    return False

print(tiene_palabra_mala("Esto es una estafa"))   # True
print(tiene_palabra_mala("Me gusta mucho"))       # False

True
False


4.2 Crea la función que decide, aplicando las dos reglas del enunciado:

In [8]:
def analizar_comentario(comentario):
    resultado = clasificador(comentario)[0]
    etiqueta = resultado["label"]
    score = resultado["score"]
    hay_palabra_mala = tiene_palabra_mala(comentario)

    if etiqueta == "NEG":
        if hay_palabra_mala or score > 0.5:
            return "RECHAZADO"
        return "ACEPTADO"
    else:
        if hay_palabra_mala:
            return "RECHAZADO"
        return "ACEPTADO"

4.3 Prueba con una lista de comentarios y muestra el resultado:

In [9]:
comentarios = [
    "Me encanta este producto",
    "El pedido llegó el martes",
    "Es una estafa",
    "No me ha gustado mucho",
]

for comentario in comentarios:
    print(comentario, "->", analizar_comentario(comentario))

Me encanta este producto -> ACEPTADO
El pedido llegó el martes -> ACEPTADO
Es una estafa -> RECHAZADO
No me ha gustado mucho -> RECHAZADO
